In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import torchvision.models as models

In [2]:
# 0. 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

########################################################
# 1. 학습/검증 데이터 전처리 파이프라인 분리 (v2 적용)
########################################################

# 1-1. 학습 데이터용 파이프라인 (ToImage 추가)
train_transform = transforms.Compose([
    transforms.ToImage(),                                # PIL 이미지를 Image Tensor로 변환
    transforms.Resize((224, 224)),                       # 224x224 리사이징
    transforms.Grayscale(num_output_channels=3),        # 1채널 -> 3채널(RGB) 변환
    transforms.RandomHorizontalFlip(p=0.5),             # 50% 확률로 좌우 반전
    transforms.RandomRotation(degrees=15),               # ±15도 범위 내 랜덤 회전
    transforms.ColorJitter(brightness=0.2, contrast=0.2),# Brightness/Contrast 변형
    transforms.ToDtype(torch.float32, scale=True),      # float32 변환 및 0~1 스케일링
    transforms.Normalize(                               # ImageNet 사전 학습 모델 표준 정규화
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
])

# 1-2. 검증 및 테스트 데이터용 파이프라인 (ToImage 추가)
val_test_transform = transforms.Compose([
    transforms.ToImage(),                                # PIL 이미지를 Image Tensor로 변환
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToDtype(torch.float32, scale=True),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
])

Using device: cpu


AttributeError: module 'torchvision.transforms' has no attribute 'ToImage'

In [ ]:
########################################################
# 2. 데이터셋 분할 및 Transform 각각 적용
########################################################

# Subset 개별 transform 적용을 위한 커스텀 Dataset 클래스
class TransformedDataset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

# 전체 데이터 로드 (원본 상태)
full_train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=val_test_transform)

# Train / Validation 분할 (55,000 / 5,000)
train_size = 55000
val_size = 5000
train_subset, val_subset = random_split(full_train_dataset, [train_size, val_size])

# 각 Split에 맞는 Transform 지정
train_dataset = TransformedDataset(train_subset, transform=train_transform)
val_dataset = TransformedDataset(val_subset, transform=val_test_transform)

# 데이터로더 구축
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


In [ ]:
from pathlib import Path

save_dir = Path("./augmented_mnist")
save_dir.mkdir(parents=True, exist_ok=True)

row, col = 2, 4

transform = transforms.Compose([
    transforms.RandomAffine(
        degrees=0.3,
        scale=(0.8, 1.2)
    )
])

fig, axes = plt.subplots(row, col, figsize=(8, 4))
axes = np.asarray(axes).reshape(row, col)

for idx in range(row * col):
    image, label = mnist_train_raw[idx]

    augmented = transform(image)

    # PIL 이미지로 저장
    save_path = save_dir / f"aug_{idx+1:02d}_label_{label}.png"
    augmented.save(save_path)

    axes[idx // col, idx % col].imshow(augmented, cmap="Greys")
    axes[idx // col, idx % col].axis("off")
    axes[idx // col, idx % col].set_title(f"label={label}")

plt.tight_layout()
plt.show()

print(f"저장 위치 : {save_dir.resolve()}")
print("저장된 파일 수 :", len(list(save_dir.glob("*.png"))))
